In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

# --- CONFIGURATION ---
# Path to chromedriver (update if needed)
CHROMEDRIVER_PATH = 'chromedriver'  # or r'C:\path\to\chromedriver.exe'
TARGET_ORGANISATION = "JUIDCO World Bank"

# --- SETUP SELENIUM ---
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
driver = webdriver.Chrome(CHROMEDRIVER_PATH, options=options)

# --- OPEN SITE ---
driver.get('https://jharkhandtenders.gov.in/nicgep/app')

# --- MANUAL LOGIN IF NEEDED ---
input("Please login manually (and solve CAPTCHA if shown). Press Enter after you are logged in and can see the tender list page...")

# --- SELECT ORGANISATION IN DROPDOWN ---
# Wait for dropdown and select organisation
org_select = driver.find_element(By.ID, "OrganName")
for option in org_select.find_elements(By.TAG_NAME, 'option'):
    if option.text.strip() == TARGET_ORGANISATION:
        option.click()
        break

# Click the 'Search' or 'Submit' button (find the exact button if required)
# You may need to inspect the actual button ID or class and update accordingly
search_button = driver.find_element(By.XPATH, "//input[@type='submit' or @type='button' and (@value='Search' or @value='Show Tenders')]")
search_button.click()
time.sleep(3)  # Wait for results to load

# --- SCRAPE TENDER TABLE ---
soup = BeautifulSoup(driver.page_source, 'html.parser')
table = soup.find('table', {'id': 'tabList'})

# Extract table rows
rows = table.find_all('tr')[1:]  # skip header row

data = []
for row in rows:
    cols = [td.get_text(strip=True) for td in row.find_all('td')]
    if len(cols) < 6:
        continue
    # Optionally extract tender detail link:
    link_tag = row.find('a', {'title': 'View Tender Status'})
    detail_url = ''
    if link_tag and 'href' in link_tag.attrs:
        detail_url = 'https://jharkhandtenders.gov.in' + link_tag['href']
    data.append({
        'S.No': cols[0],
        'Tender ID': cols[1],
        'Title and Ref.No.': cols[2],
        'Organisation Chain': cols[3],
        'Tender Stage': cols[4],
        'Status': cols[5],
        'Detail URL': detail_url
    })

# --- SAVE TO EXCEL ---
df = pd.DataFrame(data)
df.to_excel('jharkhand_tenders.xlsx', index=False)
print("Saved to jharkhand_tenders.xlsx")

driver.quit()


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"

service = Service(CHROMEDRIVER_PATH)
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
driver = webdriver.Chrome(service=service, options=options)

driver.get('https://jharkhandtenders.gov.in/nicgep/app')

print(
    "\n1. In the browser window, select the desired organization from dropdown."
    "\n2. Enter the CAPTCHA manually."
    "\n3. Click the Search/Show Tenders button manually."
    "\n4. When the tender list table is visible, come back here and press Enter to scrape."
)
input(">>> Press Enter to start scraping once results are visible...")

soup = BeautifulSoup(driver.page_source, 'html.parser')
table = soup.find('table', {'id': 'tabList'})

if not table:
    print("No tender table found. Did you click Search and see results in the browser?")
    driver.quit()
    exit()

rows = table.find_all('tr')[1:]  # skip header
all_data = []

for row in rows:
    cols = [td.get_text(strip=True) for td in row.find_all('td')]
    if len(cols) < 6:
        continue
    link_tag = row.find('a', {'title': 'View Tender Status'})
    detail_url = ''
    if link_tag and 'href' in link_tag.attrs:
        detail_url = 'https://jharkhandtenders.gov.in' + link_tag['href']
    all_data.append({
        'S.No': cols[0],
        'Tender ID': cols[1],
        'Title and Ref.No.': cols[2],
        'Organisation Chain': cols[3],
        'Tender Stage': cols[4],
        'Status': cols[5],
        'Detail URL': detail_url
    })

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel('jharkhand_tenders.xlsx', index=False)
print("\nAll data saved to 'jharkhand_tenders.xlsx'.")

driver.quit()



1. In the browser window, select the desired organization from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button manually.
4. When the tender list table is visible, come back here and press Enter to scrape.


In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"
ORG_OPTIONS = [
    ("82", "JUIDCO World Bank"),
    ("131", "Water Resources Department World Bank"),
    ("134", "WRD EinC Major-Med n Minor")
]

service = Service(CHROMEDRIVER_PATH)
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://jharkhandtenders.gov.in/nicgep/app?page=WebTenderStatusLists&service=page')

all_data = []

for value, org_name in ORG_OPTIONS:
    print(f"\n--- {org_name} ---")
    print(
        f"1. In the browser, select organisation: {org_name} from dropdown."
        "\n2. Enter the CAPTCHA manually."
        "\n3. Click the Search/Show Tenders button."
        "\n4. Wait for the results table to appear."
        "\n5. When the tender list is visible, come back here and press Enter to scrape."
    )
    input(">>> Press Enter to scrape this organisation's tenders...")

    # Parse the table after manual actions
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    table = soup.find('table', {'id': 'tabList'})

    if not table:
        print("No tender table found! Make sure results are visible in the browser before pressing Enter.")
        continue

    rows = table.find_all('tr')[1:]  # skip header
    for row in rows:
        cols = [td.get_text(strip=True) for td in row.find_all('td')]
        if len(cols) < 6:
            continue
        link_tag = row.find('a', {'title': 'View Tender Status'})
        detail_url = ''
        if link_tag and 'href' in link_tag.attrs:
            detail_url = 'https://jharkhandtenders.gov.in' + link_tag['href']
        all_data.append({
            'Organisation': org_name,
            'S.No': cols[0],
            'Tender ID': cols[1],
            'Title and Ref.No.': cols[2],
            'Organisation Chain': cols[3],
            'Tender Stage': cols[4],
            'Status': cols[5],
            'Detail URL': detail_url
        })

    print(f"Scraped {len(rows)} rows for {org_name}.\n")
    # Optional: clear search/filter in browser for next organization

df = pd.DataFrame(all_data)
df.to_excel('jharkhand_tenders_ALL.xlsx', index=False)
print("\nAll data saved to 'jharkhand_tenders_ALL.xlsx'.")

driver.quit()



--- JUIDCO World Bank ---
1. In the browser, select organisation: JUIDCO World Bank from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button.
4. Wait for the results table to appear.
5. When the tender list is visible, come back here and press Enter to scrape.


>>> Press Enter to scrape this organisation's tenders... 


Scraped 9 rows for JUIDCO World Bank.


--- Water Resources Department World Bank ---
1. In the browser, select organisation: Water Resources Department World Bank from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button.
4. Wait for the results table to appear.
5. When the tender list is visible, come back here and press Enter to scrape.


>>> Press Enter to scrape this organisation's tenders... 


Scraped 12 rows for Water Resources Department World Bank.


--- WRD EinC Major-Med n Minor ---
1. In the browser, select organisation: WRD EinC Major-Med n Minor from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button.
4. Wait for the results table to appear.
5. When the tender list is visible, come back here and press Enter to scrape.


>>> Press Enter to scrape this organisation's tenders... 


Scraped 12 rows for WRD EinC Major-Med n Minor.


All data saved to 'jharkhand_tenders_ALL.xlsx'.


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"
ORG_OPTIONS = [
    ("82", "JUIDCO World Bank"),
    ("131", "Water Resources Department World Bank"),
    ("134", "WRD EinC Major-Med n Minor")
]

service = Service(CHROMEDRIVER_PATH)
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://jharkhandtenders.gov.in/nicgep/app?page=WebTenderStatusLists&service=page')

all_data = []

for value, org_name in ORG_OPTIONS:
    print(f"\n--- {org_name} ---")
    print(
        f"1. In browser, select organisation: {org_name} from dropdown."
        "\n2. Enter the CAPTCHA manually."
        "\n3. Click the Search/Show Tenders button."
        "\n4. Wait for the results table to appear."
        "\n5. When the tender list is visible, come back here and press Enter to scrape all pages."
    )
    input(">>> Press Enter to start scraping this organisation (all pages)...")

    while True:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table', {'id': 'tabList'})

        if not table:
            print("No tender table found! Make sure results are visible in the browser before pressing Enter.")
            break

        rows = table.find_all('tr')[1:]  # skip header
        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all('td')]
            if len(cols) < 6:
                continue
            link_tag = row.find('a', {'title': 'View Tender Status'})
            detail_url = ''
            if link_tag and 'href' in link_tag.attrs:
                detail_url = 'https://jharkhandtenders.gov.in' + link_tag['href']
            all_data.append({
                'Organisation': org_name,
                'S.No': cols[0],
                'Tender ID': cols[1],
                'Title and Ref.No.': cols[2],
                'Organisation Chain': cols[3],
                'Tender Stage': cols[4],
                'Status': cols[5],
                'Detail URL': detail_url
            })

        # Try to find the "Next" button
        try:
            next_btn = driver.find_element('id', 'loadNext')
            if next_btn.is_displayed() and next_btn.is_enabled():
                next_btn.click()
                time.sleep(2)
            else:
                print(f"Reached last page for {org_name}.")
                break
        except Exception:
            print(f"Reached last page for {org_name}.")
            break

df = pd.DataFrame(all_data)
df.to_excel('jharkhand_tenders_ALL_orgs_ALLpages.xlsx', index=False)
print("\nAll data saved to 'jharkhand_tenders_ALL_orgs_ALLpages.xlsx'.")

driver.quit()



--- JUIDCO World Bank ---
1. In browser, select organisation: JUIDCO World Bank from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button.
4. Wait for the results table to appear.
5. When the tender list is visible, come back here and press Enter to scrape all pages.


>>> Press Enter to start scraping this organisation (all pages)... 


Reached last page for JUIDCO World Bank.

--- Water Resources Department World Bank ---
1. In browser, select organisation: Water Resources Department World Bank from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button.
4. Wait for the results table to appear.
5. When the tender list is visible, come back here and press Enter to scrape all pages.


>>> Press Enter to start scraping this organisation (all pages)... 


Reached last page for Water Resources Department World Bank.

--- WRD EinC Major-Med n Minor ---
1. In browser, select organisation: WRD EinC Major-Med n Minor from dropdown.
2. Enter the CAPTCHA manually.
3. Click the Search/Show Tenders button.
4. Wait for the results table to appear.
5. When the tender list is visible, come back here and press Enter to scrape all pages.


>>> Press Enter to start scraping this organisation (all pages)... 
